# Paper-exact reproduction: the Addition circuit on Qwen3-4B

**v3 (clean-room rebuild, 2026-07-14).** Reproduces the *Addition* case study of
[On the Biology of a Large Language Model](https://transformer-circuits.pub/2025/attribution-graphs/biology.html)
on Qwen3-4B with mwhanna per-layer transcoders, following `multilingual.ipynb`'s template:
**selection lives in the reviewed artifact** `supernodes/addition_4b.json`, hand-curated in the
**grid-enabled explorer** (every feature node shows its operand plots) and ingested by
`build_supernodes.py --from-exports`; this notebook only *drives* those supernodes.

The paper's claims under reproduction: the model answers `calc: a+b=` through **parallel
pathways** — input-digit features (`_6`, `_9`), low-precision **magnitude** features, a
high-precision **lookup table** (`_6+_9`), and **sum** features (`sum = _5`, `sum ~95`) —
identified by operand-plot geometry, validated by **suppression** and by **substituting**
the lookup with a `_9+_9` donor (ones digit 5 → 8); the same lookup features **fire in
prose** (a citation-style prompt) and the model's **self-report** describes the carry
algorithm the graphs do not show.

Qwen3 facts that adapt the protocol (see `addition_helper.py`): digits tokenize one by one,
so the answer has a **first-digit** moment (`…=`) and a teacher-forced **ones-digit** moment
(`…=9`); Qwen3-4B misses the paper's 36+59, so the studied pair is the nearest correctly
answered pair of the same ones-digit class (recorded in the dumps' manifest). Steering uses
the additive convention `m = M_paper − 1`.

Run on a GPU node: `sbatch hpc/run_addition_notebook.sbatch` (inputs from
`sbatch hpc/run_supernode_inputs.sbatch`, review via `build_supernodes.py`).


## What actually runs — inputs, supernodes, artifacts

**Graphs (loaded from the reviewed dumps, `supernode_inputs/4b/`):**

| dump | prompt | analyzed moment |
|---|---|---|
| `first` | `calc: a+b=` | the sum's first digit at `=` |
| `ones` | `calc: a+b=<leading digits>` | the teacher-forced ones digit |
| `donor` | the `_9+_9`-class pair, ones moment | donor lookup features for §D |
| `reuse` | citation-style prose, ones moment | the same fact in context (§E) |

**Supernodes** come exclusively from `supernodes/addition_4b.json` (approved =
hand-reviewed; the loader refuses anything else). Canonical names are pair-parametric
(`build_supernodes.addition_names`): `input _6`/`input _9` (sources at the operand ones
digits), `magnitude ~a`/`~b` + `add ~b (function)` + `magnitude lookup` + `sum ~s (low
precision)` (the first-digit magnitude path), `lookup (_6+_9)` + `sum = _5` (the ones
modular path), `lookup (_9+_9) donors`, `reuse lookups`. Interventions are **constrained
patching** (`patch_end_layer` swept and pinned per experiment; attention frozen), the
paper's protocol.

**Artifacts** land in `artifacts/paper_addition/4b/`: explorer HTMLs with grids + reviewed
groups, `grids_supernodes.png`, `behavior.json`, `suppressions.json`, `swap_sweep.{json,png}`,
`reuse.json`, `introspection.json`, `corpus_scan.json`, `verdicts.json`.


In [ ]:
%matplotlib inline
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

sys.path.insert(0, str(Path.cwd()))
import addition_helper as A
import build_supernodes as BS

from llm_circuits.circuits.graph_explorer import render_graph_explorer_html
from llm_circuits.circuits.interventions import run_feature_intervention
from llm_circuits.models.qwen3 import load_qwen3
from llm_circuits.settings import artifacts_dir
from llm_circuits.transcoders.circuit_tracer_loader import load_transcoder
from llm_circuits.utils import seed_everything

SIZE = "4b"
seed_everything(0, deterministic=True)
SN_INPUTS = artifacts_dir() / "supernode_inputs" / SIZE
OUT = artifacts_dir() / "paper_addition" / SIZE
OUT.mkdir(parents=True, exist_ok=True)

manifest = json.loads((SN_INPUTS / "manifest.json").read_text())
mg = manifest["graphs"]
A0, B0 = manifest["addition"]["pair"]
DA, DB = manifest["addition"]["donor_pair"]
S0 = A0 + B0
RA, RB, O0 = A0 % 10, B0 % 10, S0 % 10
NAMES = BS.addition_names(manifest)
SN = A.load_supernodes(Path("supernodes") / f"addition_{SIZE}.json")
missing = [n for n in NAMES if n not in SN or not SN[n]["members"]]
print(f"pair {A0}+{B0}={S0} ({manifest['addition']['pair_note']}); donor {DA}+{DB}")
print(f"supernodes: {len(SN)} loaded; canonical empty/missing: {missing or 'none'}")

device = "cuda" if torch.cuda.is_available() else "cpu"
model, tokenizer = load_qwen3(SIZE, dtype_str="bf16", device_map=device)
model.eval()
tc = load_transcoder(
    f"qwen3-{SIZE}", device=device, dtype=torch.bfloat16, lazy_decoder=False
).transcoder
ctx = BS.load_grid_ctx(SN_INPUTS)
GRAPH_NAMES = ("first", "ones", "donor", "reuse")
graphs = {g: json.loads((SN_INPUTS / f"graph_{g}.json").read_text()) for g in GRAPH_NAMES}
fin = {g: mg[g]["final_position"] for g in GRAPH_NAMES}
ans_id = {g: mg[g]["answer_token_id"] for g in GRAPH_NAMES}
ids_of = {g: A.tokenize_raw(tokenizer, mg[g]["prompt"], device) for g in GRAPH_NAMES}
for g in GRAPH_NAMES:
    assert ids_of[g].shape[1] == mg[g]["n_tokens"], f"{g}: tokenization drifted vs the dumps"

## 0. Behavior — accuracy over all 10,000 problems, the studied pair, the prompts

The paper's model answers `calc: 36+59=` correctly; Qwen3-4B's competence is patchier, so
the reproduction first maps it (greedy decoding over every `a+b`, a,b ∈ 0..99) and studies
the nearest correct pair of the paper's ones-digit class (`_6+_9`). The donor pair is the
nearest correct `_9+_9` problem (its true ones digit is 8 — the §D substitution target).


In [ ]:
if (SN_INPUTS / "accuracy_correct.npy").exists():
    acc = np.load(SN_INPUTS / "accuracy_correct.npy")
else:  # the dump job reused a recorded pair; recompute for the figure (~2 min)
    acc, _ = A.accuracy_grid(model, tokenizer)
ax = A.plot_accuracy(acc)
ax.scatter([B0], [A0], s=70, facecolors="none", edgecolors="#00e676", lw=1.6)
ax.figure.savefig(OUT / "accuracy_grid.png", dpi=150, bbox_inches="tight")

behavior = {"accuracy_pct": round(float(acc.mean()) * 100, 2)}
for g in GRAPH_NAMES:
    _, cont = A.greedy_answer(model, tokenizer, ids_of[g], n_gen=4)
    expected = mg[g].get("expected_answer", mg[g]["answer"])
    behavior[g] = {"prompt": mg[g]["prompt"], "continuation": cont, "expected": expected}
    print(f"{g:6s} {mg[g]['prompt']!r:>60s} -> {cont!r} (expected {expected!r})")
(OUT / "behavior.json").write_text(json.dumps(behavior, indent=1, ensure_ascii=False))

## A. Attribution graphs — the four dumped circuits, explorer HTMLs with grids + groups

Graphs were built and pruned once by `build_supernode_inputs.py` (recorded thresholds in
the manifest); here they are re-exported as **grid-enabled explorer pages with the
reviewed supernodes pre-loaded as groups** — the same view the selection was made in.


In [ ]:
def groups_for(gname):
    specs = []
    for name, sn in SN.items():
        if sn["graph"] != gname or not sn["members"]:
            continue
        pos = sn["position"]
        specs.append(
            {
                "name": name,
                "members": [(m["layer"], m["feature"]) for m in sn["members"]],
                "position": pos if pos == "final" or isinstance(pos, int) else None,
            }
        )
    return specs


for g in GRAPH_NAMES:
    n_feat = sum(1 for n in graphs[g]["nodes"] if n["node_type"] == "feature")
    wired = BS.attach_operand_grids(graphs[g], g, mg[g], ctx)
    render_graph_explorer_html(
        wired,
        OUT / f"graph_{g}.html",
        labels=[g],
        title=f"addition {g} ({A0}+{B0})",
        groups=groups_for(g),
    )
    print(f"{g:6s} {n_feat:5d} feature nodes, {len(groups_for(g)):2d} groups -> graph_{g}.html")

## B. Supernodes — the reviewed selection and its operand-grid evidence

One operand plot per member (the probe matching the supernode's moment), ringed at the
studied pair — the paper's feature-identity argument in one figure. Sum-side groups also
get their **direct decoder weights** on the ten digit tokens (`sum = _5` should write "5").


In [ ]:
print(f"{'supernode':32s} {'role':10s} {'graph':6s} {'pos':6s} {'n':>2s}  grid classes")
for name, sn in SN.items():
    classes = sorted({str((m.get("evidence") or {}).get("grid_class")) for m in sn["members"]})
    print(
        f"{name:32s} {sn['role']:10s} {sn['graph']:6s} {sn['position']!s:6s} "
        f"{len(sn['members']):2d}  {'; '.join(classes)[:70]}"
    )

items = []
for name, sn in SN.items():
    if not sn["members"]:
        continue
    target = str(mg[sn["graph"]].get("target", ""))
    probe = (
        ("ones" if target in ("ones", "reuse-ones") else "final")
        if sn["position"] == "final"
        else "peak"
    )
    for m in sn["members"]:
        g = ctx.grid(probe, m["layer"], m["feature"])
        if g is not None:
            items.append((g, f"{name[:16]}\nL{m['layer']}f{m['feature']} [{probe}]"))
fig = A.plot_grid_panels(items, ncols=6, mark=(A0, B0))
fig.savefig(OUT / "grids_supernodes.png", dpi=130, bbox_inches="tight")
print(f"grids_supernodes.png: {len(items)} member grids")

sum_groups = [f"sum = _{O0}", f"sum ~{S0} (low precision)"]
sum_feats = [(m["layer"], m["feature"]) for n in sum_groups if SN.get(n) for m in SN[n]["members"]]
weights = A.digit_direct_weights(model, tc, sum_feats, tokenizer) if sum_feats else {}
for (layer, feat), w in weights.items():
    top = int(np.argmax(w))
    print(f"L{layer:2d}f{feat:6d} direct digit weights: argmax '{top}' ({w[top]:+.3f})")
(OUT / "direct_weights.json").write_text(
    json.dumps({f"L{L}f{f}": w for (L, f), w in weights.items()}, indent=1)
)

## C. Suppression interventions — each pathway's causal role (paper's inhibition tests)

Each source supernode is suppressed at its members' own node positions under constrained
patching, at two strengths: **ablate** (`M=0`, `m=−1`) and the paper's **sign flip**
(`M=−1`, `m=−2`). The patch end layer ℓ is chosen per intervention by the paper's recipe
(sweep ℓ, keep the most-suppressing). Reported per run: the answer-token shift, the
**ones-digit distribution** and its smear width, and %-readouts of the *downstream*
supernodes (per-feature ratio, pinned members excluded) — the paper's arrows:
inputs → lookup → sum, magnitudes → sum-band.


In [ ]:
def downstream_of(gname, exclude):
    return {
        n: (sn, "baseline")
        for n, sn in SN.items()
        if sn["graph"] == gname and sn["members"] and n != exclude
    }


SUPPRESS = [
    n
    for n in (
        f"input _{RA}",
        f"input _{RB}",
        f"lookup (_{RA}+_{RB})",
        f"magnitude ~{A0}",
        f"magnitude ~{B0}",
    )
    if SN.get(n, {}).get("members")
]
suppress_results = {}
for name in SUPPRESS:
    g = SN[name]["graph"]
    tok = tokenizer.decode([ans_id[g]])
    for mult in (0.0, -1.0):
        ivs = A.suppress_ivs(SN[name], mult=mult, final_pos=fin[g])
        ell, sweep = A.choose_end_layer(model, tc, ids_of[g], ivs, ans_id[g], mode="suppress")
        rep = A.steer_report(
            model,
            tc,
            tokenizer,
            ids_of[g],
            ivs,
            patch_end_layer=ell,
            readout_sns=downstream_of(g, name),
            final_pos=fin[g],
        )
        rep["graph"], rep["mult"], rep["ell"] = g, mult, ell
        rep["p_answer_before"] = sweep.baseline_prob
        rep["p_answer_after"] = sweep.probs[sweep.end_layers.index(ell)]
        suppress_results[f"{name} @ M={mult:+.0f}"] = rep
        ro = {
            n: r["mean_pct"]
            for n, r in rep["readouts"].items()
            if r["mean_pct"] is not None and r["n_used"]
        }
        print(
            f"{name:26s} M={mult:+.0f} ell={ell:2d}  p({tok.strip()!r}) "
            f"{rep['p_answer_before']:.3f}->{rep['p_answer_after']:.3f}  "
            f"top after: {rep['top_after'][0]}  width {rep['width_before']}->"
            f"{rep['width_after']}  readouts {ro}"
        )
(OUT / "suppressions.json").write_text(json.dumps(suppress_results, indent=1, ensure_ascii=False))

## D. The lookup substitution — `_6+_9` → `_9+_9` retargets the ones digit (paper: 5 → 8)

The paper's sharpest causal test: on the ones-digit moment, flip the studied lookup
supernode and inject the **donor problem's** lookup features at their donor-prompt
activations. The coupled ramp hits the substitution endpoint at s = 1 (source `M=−1`,
donor ×1); ℓ is chosen at the endpoint by the most-promoting rule for the expected digit.


In [ ]:
lookup_name, donor_name = f"lookup (_{RA}+_{RB})", f"lookup (_{DA % 10}+_{DB % 10}) donors"
expected_digit = str((DA + DB) % 10)
expected_id = tokenizer(expected_digit, add_special_tokens=False).input_ids[0]
baseline_id = ans_id["ones"]


def swap_ivs(s):
    if s == 0:
        return []
    sup = A.suppress_ivs(SN[lookup_name], mult=1.0 - 2.0 * s, final_pos=fin["ones"])
    inj, _used = A.inject_ivs(SN[donor_name], mult=s, positions=[fin["ones"]])
    return sup + inj


ell, endpoint_sweep = A.choose_end_layer(
    model, tc, ids_of["ones"], swap_ivs(1.0), expected_id, mode="promote"
)
ladder = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0]
swap = {"ell": ell, "ladder": ladder, "p_base": [], "p_exp": [], "top": []}
for s in ladder:
    ivs = swap_ivs(s)
    res = run_feature_intervention(
        model,
        tc,
        ids_of["ones"],
        ivs,
        n_bos_tokens=A.N_BOS,
        patch_end_layer=ell if ivs else None,
    )
    probs = res.ablated_logits[-1].float().softmax(-1)
    swap["p_base"].append(float(probs[baseline_id]))
    swap["p_exp"].append(float(probs[expected_id]))
    swap["top"].append(A.top_token_probs(res.ablated_logits[-1], tokenizer, 4))
swap["crossover"] = next(
    (s for s, pb, pe in zip(ladder, swap["p_base"], swap["p_exp"], strict=True) if pe > pb),
    None,
)
print(f"ell={ell}; crossover at s={swap['crossover']} (paper endpoint s=1)")
for s, pb, pe, top in zip(ladder, swap["p_base"], swap["p_exp"], swap["top"], strict=True):
    print(f"  s={s:4.2f}  p({O0})={pb:.3f}  p({expected_digit})={pe:.3f}  top {top[:2]}")

fig, ax = plt.subplots(figsize=(4.6, 3.2))
ax.plot(ladder, swap["p_base"], "o-", label=f"p('{O0}') baseline answer")
ax.plot(ladder, swap["p_exp"], "s-", label=f"p('{expected_digit}') donor answer")
ax.axvline(1.0, color="#999", ls="--", lw=1, label="paper endpoint (s=1)")
ax.set_xlabel("substitution strength s")
ax.set_ylabel("probability")
ax.set_title(f"lookup swap _{RA}+_{RB} -> _{DA % 10}+_{DB % 10} (ell={ell})", fontsize=9)
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUT / "swap_sweep.png", dpi=150)
(OUT / "swap_sweep.json").write_text(json.dumps(swap, indent=1))

## E. Reuse in context — the same lookup fires (and acts) in prose

The citation-style prompt encodes the same ones-digit fact ("founded in 19{b}, volume {a}
→ 19{a+b}"). Checks: (i) behavior; (ii) the calc lookup members' activations at the reuse
ones moment (from the dumps); (iii) suppressing the reviewed `reuse lookups` group moves
the prose answer too.


In [ ]:
reuse = {
    "prompt": mg["reuse"]["prompt"],
    "behavior_ok": bool(mg["reuse"].get("behavior_ok")),
    "continuation": behavior["reuse"]["continuation"],
}
acts = json.loads((SN_INPUTS / "reuse_moment_acts.json").read_text())
active = {(layer, feat): a for layer, feat, a in acts["active_final"]}
reuse["lookup_acts"] = []
for m in SN[lookup_name]["members"]:
    a = active.get((m["layer"], m["feature"]), 0.0)
    reuse["lookup_acts"].append([m["layer"], m["feature"], m["act"], a])
    print(
        f"lookup L{m['layer']:2d}f{m['feature']:6d}  calc act {m['act']:6.2f}   "
        f"reuse-moment act {a:6.2f}"
    )

rg = SN.get(f"reuse lookups (_{RA}+_{RB})")
if rg and rg["members"]:
    ivs = A.suppress_ivs(rg, mult=-1.0, final_pos=fin["reuse"])
    ell_r, _ = A.choose_end_layer(model, tc, ids_of["reuse"], ivs, ans_id["reuse"], mode="suppress")
    rep = A.steer_report(
        model, tc, tokenizer, ids_of["reuse"], ivs, patch_end_layer=ell_r, final_pos=fin["reuse"]
    )
    reuse["suppression"] = {
        "ell": ell_r,
        "top_before": rep["top_before"][:3],
        "top_after": rep["top_after"][:3],
    }
    print(
        f"suppress reuse lookups @ M=-1, ell={ell_r}: {rep['top_before'][0]} -> {rep['top_after'][0]}"
    )
else:
    reuse["suppression"] = None
    print("no reviewed reuse-lookup group — suppression skipped (documented absence)")
(OUT / "reuse.json").write_text(json.dumps(reuse, indent=1, ensure_ascii=False))

## F. Introspection — the model's self-report vs the mechanism

The paper asks "Briefly, how did you get that?" and gets the schoolbook carry algorithm —
not the lookup/magnitude pathways the graphs show ("a capability without metacognitive
insight into it").


In [ ]:
intro = A.introspection_dialogue(model, tokenizer, A0, B0)
print(f"Q: {intro['question']}\nA: {intro['answer']}\nQ: {intro['how']}\nA: {intro['explanation']}")
algo_words = ("carry", "ones", "tens", "units", "add the", "then add", "column")
intro["mentions_algorithm"] = any(w in intro["explanation"].lower() for w in algo_words)
intro["mentions_lookup"] = "lookup" in intro["explanation"].lower()
(OUT / "introspection.json").write_text(json.dumps(intro, indent=1, ensure_ascii=False))

## G. Corpus scan — where the circuit's features fire in the wild

The paper finds the lookup/sum features on citations, running totals, and schedule
arithmetic. A small fixed text battery (offline, deterministic) is scanned for the
reviewed lookup + sum members; every activation above threshold is recorded with its
token window.


In [ ]:
TEXTS = [
    "References: 12. Smith, J. et al., Polymer, vol. 46, pp. 1949-1962 (1995).",
    "The journal was founded in 1949; its 46th volume therefore appeared in 1995.",
    "Invoice subtotal: $46.00, shipping: $49.00, total due: $95.00.",
    "He scored 46 in the first innings and 49 in the second, 95 runs in the match.",
    "The 46 bus leaves at minute 49 of every hour, arriving 95 minutes later.",
    "Room capacity is 46 seats plus 49 standing, 95 people at most.",
    "Chapter 46 ends on page 149; chapter 47 starts on page 155.",
    "In 1949 the population was 46 thousand; by 1995 it had tripled.",
    "Add 46 grams of flour and 49 grams of sugar for 95 grams of dry mix.",
    "The odometer read 12,346 before the 49-mile trip: 12,395 after.",
    "Flight AA46 departs gate 49 at 09:50.",
    "temperatures reached 46C in the shade and 49C on the tarmac.",
    "for i in range(46, 95): total += weights[i - 49]",
    "The committee of 46 senators and 49 representatives totals 95 members.",
    "It rained all week, and the garden had never looked greener.",
]
scan_feats = sorted(
    {
        (m["layer"], m["feature"])
        for n in (lookup_name, f"sum = _{O0}")
        if SN.get(n)
        for m in SN[n]["members"]
    }
)
hits = A.corpus_scan(model, tc, tokenizer, scan_feats, TEXTS, act_floor=0.5)
n_hits = sum(len(v) for v in hits.values())
print(f"{n_hits} activations > 0.5 across {len(TEXTS)} texts / {len(scan_feats)} features")
for (layer, feat), lst in sorted(hits.items()):
    for h in lst[:2]:
        print(f"L{layer:2d}f{feat:6d} act {h['act']:5.2f}  ...{h['snippet']}...")
(OUT / "corpus_scan.json").write_text(
    json.dumps(
        {"texts": TEXTS, "hits": {f"L{L}f{f}": v for (L, f), v in hits.items()}},
        indent=1,
        ensure_ascii=False,
    )
)

## Summary — verdicts vs the paper

Programmatic verdicts from the runs above (criteria in code; the table is regenerated on
every execution). *Reproduced* = the paper's qualitative claim holds at its protocol
strengths on Qwen3-4B; *partial* = holds with caveats (weaker strength, subset of
members); *not reproduced* = the measured outcome contradicts the claim.


In [ ]:
verdicts = []


def verdict(claim, outcome, result):
    verdicts.append({"claim": claim, "outcome": outcome, "result": result})


verdict(
    "behavior: two-digit addition",
    f"{behavior['accuracy_pct']}% greedy accuracy; studied pair {A0}+{B0} "
    f"({manifest['addition']['pair_note']})",
    "reproduced (adapted pair)" if behavior["accuracy_pct"] > 50 else "partial",
)
lookup_ok = bool(SN.get(lookup_name, {}).get("members"))
sum_ok = bool(SN.get(f"sum = _{O0}", {}).get("members"))
mag_ok = any(SN.get(f"magnitude ~{x}", {}).get("members") for x in (A0, B0))
verdict(
    "feature families (operand plots): inputs, magnitudes, lookup, sums",
    f"reviewed groups non-empty: lookup={lookup_ok}, sum={sum_ok}, magnitude={mag_ok}, "
    f"inputs={bool(SN.get(f'input _{RA}', {}).get('members'))}",
    "reproduced" if (lookup_ok and sum_ok and mag_ok) else "partial",
)
flips = {
    k: r["top_after"][0][0] != tokenizer.decode([ans_id[r["graph"]]]).strip()
    for k, r in suppress_results.items()
    if r["mult"] == -1.0
}
verdict(
    "suppressing each pathway moves the answer",
    "; ".join(f"{k.split(' @')[0]}: {'moved' if v else 'held'}" for k, v in flips.items()),
    "reproduced"
    if flips and all(flips.values())
    else ("partial" if any(flips.values()) else "not reproduced"),
)
cx = swap["crossover"]
verdict(
    f"lookup substitution retargets the ones digit ({O0} -> {expected_digit})",
    f"crossover at s={cx} (paper endpoint s=1), ell={swap['ell']}",
    "reproduced"
    if cx is not None and cx <= 1.0
    else (f"partial (needs {cx}x)" if cx is not None else "not reproduced"),
)
reuse_fires = any(a > 0.5 for *_x, a in reuse["lookup_acts"])
reuse_moves = bool(reuse.get("suppression")) and (
    reuse["suppression"]["top_after"][0][0] != reuse["suppression"]["top_before"][0][0]
)
verdict(
    "the lookup features fire and act in prose (citation context)",
    f"behavior_ok={reuse['behavior_ok']}, fires={reuse_fires}, "
    f"suppression moves answer={reuse_moves}",
    "reproduced"
    if (reuse["behavior_ok"] and reuse_fires and reuse_moves)
    else ("partial" if reuse_fires else "not reproduced"),
)
verdict(
    "self-report describes the carry algorithm, not the mechanism",
    f"mentions algorithm={intro['mentions_algorithm']}, mentions lookup={intro['mentions_lookup']}",
    "reproduced" if intro["mentions_algorithm"] and not intro["mentions_lookup"] else "partial",
)

w = max(len(v["claim"]) for v in verdicts)
for v in verdicts:
    print(f"{v['claim']:{w}s} | {v['result']:24s} | {v['outcome'][:90]}")
(OUT / "verdicts.json").write_text(json.dumps(verdicts, indent=1, ensure_ascii=False))
print(f"\nartifacts -> {OUT}")